In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed Nov  3 18:41:41 2021

@author: DELL
"""

import  numpy  as  np
import libpysal as ps
from stwr.gwr import GWR, MGWR,STWR
from stwr.sel_bw import *
from stwr.utils import shift_colormap, truncate_colormap
import geopandas as gp
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import math
from matplotlib.gridspec import GridSpec
import time
import csv 
import copy 
import rasterio
import rasterio.plot
import rasterio.features
import rasterio.warp
#import pyproj
from scipy import stats
#导入时间处理工具
import time
import datetime
#from datetime import datetime
from datetime import date
import os

from spglm.family import Gaussian, Binomial, Poisson
from spglm.glm import GLM, GLMResults

if __name__ == '__main__':

    def extract_str_filedate(filename):
        datetime_string = filename.replace('.csv', '')
        return datetime_string

    def extract_datetime(filename):
        datetime_string = filename.replace('.csv', '')
        return datetime.datetime.strptime(datetime_string,'%Y%m%d')

    def extract_str_datetime(strVal):
        return datetime.datetime.strptime(strVal,'%Y%m%d')
    ##########################################################################

    #def extract_mouthtime(filename):
    #    monthtime_string = filename.replace('.csv', '')
    #    return datetime.datetime.strptime(monthtime_string,'%Y%m')

    #def extract_str_mounthtime(strVal):
    #    return datetime.datetime.strptime(strVal,'%Y%m')

    ##########################################################################
    #bbb = extract_datetime('20130102')

    # '2015-08-28 16:43:37.283' --> 1440751417.283
    # 或者 '2015-08-28 16:43:37' --> 1440751417.0
    def string2timestamp(strValue):
      try:
        d = datetime.datetime.strptime(strValue, "%Y%m%d")
        t = d.timetuple()
        timeStamp = int(time.mktime(t))
        timeStamp = float(str(timeStamp) + str("%06d" % d.microsecond))/1000000
        #print(timeStamp)
        return timeStamp
      except ValueError as e:
        print (e)
    #    d = datetime.datetime.strptime(str2, "%Y%m%d")
    #    t = d.timetuple()
    #    timeStamp = int(time.mktime(t))
    #    timeStamp = float(str(timeStamp) + str("%06d" % d.microsecond))/1000000
    #    print (timeStamp)
        return timeStamp


    #def string2timestampf(dtime):
    #  try:
    #    d = dtime.strptime(strValue, "%Y%m%d")
    #    t = d.timetuple()
    #    timeStamp = int(time.mktime(t))
    #    timeStamp = float(str(timeStamp) + str("%06d" % d.microsecond))/1000000
    #    print(timeStamp)
    #    return timeStamp
    #  except ValueError as e:
    #    print (e)
    #    d = datetime.datetime.strptime(str2, "%Y%m%d")
    #    t = d.timetuple()
    #    timeStamp = int(time.mktime(t))
    #    timeStamp = float(str(timeStamp) + str("%06d" % d.microsecond))/1000000
    #    print (timeStamp)
    #    return timeStamp

    # 1440751417.283 --> '2015-08-28 16:43:37.283'
    def timestamp2string(timeStamp):
      try:
        d = datetime.datetime.fromtimestamp(timeStamp)
        str1 = d.strftime("%Y%m%d")
        # 2015-08-28 16:43:37.283000'
        return str1
      except Exception as e:
        print (e)
        return ''

    AICc_out=[]
    R2_out = []
    RSS_out = []
    Sigma_out = []
    output_filepath = "D:\\STWR\\Data_STWR\\liuganoutput02\\"
    for root,dirs,files in os.walk("D:\\STWR\\Data_STWR\\liuganinput\\"):
        for dir in dirs:
            FilesPath = os.path.join(root,dir)+'/'
            file_loads = os.listdir(FilesPath)
            files_len = len(file_loads)
            cal_files = file_loads[:files_len]
          #  for size_len in range(files_len):
               # cal_files = file_loads[:(size_len+1)]
            size_len = 0
            Ename = None
            for filename in cal_files:
                str_full_filename = FilesPath + filename
                csvFile = open(str_full_filename, "r",encoding='UTF-8' )#,encoding='UTF-8'
                print(str_full_filename)
                df = pd.read_csv(csvFile,usecols=['cal_X','cal_y','ename','y','Y1',
                                                  'Y2','Y3','Y4','time'])
                # #标准化
                cols = list(['y','Y1','Y2','Y3','Y4'])  #输入需要标准化的列名
                for item in cols:
                    mean_tmp = np.mean(np.array(df[item]))
                    std_tmp = np.std(np.array(df[item]))
                    if (std_tmp):
                        df[item] = df[item].apply(lambda x: (x - mean_tmp) / std_tmp)
                #选择csv中要写入的所有数据的列
                #all_data =  df.values
                ####################################################################
                #tick_time = all_data[0,-1]
                #tick_timestamp = timestamp2string(tick_time)
               # tick_ext_time = extract_str_datetime(tick_timestamp)

                cal_coords_list =[]
                cal_y_list =[]
                cal_X_list =[]
                delt_stwr_intervel =[0.0]
                
                     
               #####################################
                if Ename is None:
                    Ename = df.values[:,2]
                df = df.drop(['ename'], axis=1)    
                df = df.sort_values(by=['time'])  
                all_data = df.values
                tick_time = all_data[0,-1]
                cal_coord_tick = []
                cal_X_tick =[]
                cal_y_tick =[]
        
                time_tol = 1.0e-7
            
                lensdata = len(all_data)
                for row in range(lensdata):
                    cur_time = all_data[row,-1]
                    if(abs(cur_time-tick_time)>time_tol):
                        cal_coords_list.append(np.asarray(cal_coord_tick))
                        cal_X_list.append(np.asarray(cal_X_tick))
                        cal_y_list.append(np.asarray(cal_y_tick))
                        delt_t = cur_time - tick_time
                        delt_stwr_intervel.append(delt_t) 
                        tick_time =cur_time
                        cal_coord_tick = []
                        cal_X_tick =[]
                        cal_y_tick =[]
                    coords_tick = np.array([all_data[row,0],all_data[row,1]])
                    cal_coord_tick.append(coords_tick)
            
                    x_tick = np.array([all_data[row,3],all_data[row,4],all_data[row,5],all_data[row,6]])
                    cal_X_tick.append(x_tick)
                    y_tick = np.array([all_data[row,2]])
                    cal_y_tick.append(y_tick)
               #####################################
               
                #gwr
                cal_cord_gwr = np.asarray(cal_coord_tick)
                cal_X_gwr  = np.asarray(cal_X_tick)
                cal_y_gwr = np.asarray(cal_y_tick)
                cal_coords_list.append(np.asarray(cal_coord_tick))
                cal_X_list.append(np.asarray(cal_X_tick))
                cal_y_list.append(np.asarray(cal_y_tick))
                #A = cal_y_list[-1]
                #B = cal_X_list[-1]
                #OLS
                ols_result = GLM(cal_y_list[-1],cal_X_list[-1],constant=False,family=Gaussian()).fit()
    #            for i in range(ols_X.shape[1]):
    #                print(ols_result.params[i])
                #ols_predbeta_results.append(ols_result.params)
                #stwr
                stwr_selector_ = Sel_Spt_BW(cal_coords_list, cal_y_list, cal_X_list,delt_stwr_intervel ,spherical = True)
                optalpha,optsita,opt_btticks,opt_gwr_bw0 = stwr_selector_.search()
                #调整不要将opt_btticks减1
                # = STWR(cal_coords_list,cal_y_list,cal_X_list,delt_stwr_intervel,optsita,opt_gwr_bw0,tick_nums=opt_btticks,alpha =optalpha,spherical = True,recorded=1)
                #stwr_model = STWR(cal_coords_list, cal_y_list, cal_X_list, delt_stwr_intervel, optsita, opt_gwr_bw0,
                                  #tick_nums=opt_btticks + 1, alpha=optalpha, recorded=1)
                stwr_model = STWR(cal_coords_list,cal_y_list,cal_X_list,delt_stwr_intervel,optsita,opt_gwr_bw0,alpha =optalpha,spherical = True,recorded=1)
                #F-STWR
                #stwr
                stwr_results = stwr_model.fit()
            #    print(stwr_results.summary())
                stwr_scale = stwr_results.scale
                stwr_residuals = stwr_results.resid_response
            #    stwr_residuals = np.asarray(stwr_residuals)
            #    stwr_residuals.reshape((-1,1))
            #    stwr_residuals

                stwr_beta_se = stwr_results.bse
                stwr_r2 = stwr_results.localR2

                #gwr
                gwr_selector = Sel_BW(cal_cord_gwr, cal_y_gwr, cal_X_gwr,spherical = True)
                gwr_bw= gwr_selector.search()
                gwr_model = GWR(cal_cord_gwr, cal_y_gwr, cal_X_gwr, gwr_bw,spherical = True)
                gwr_results = gwr_model.fit()
            #    print(gwr_results.summary())
                gw_rscale = gwr_results.scale
                gwr_residuals = gwr_results.resid_response
            #    gwr_residuals = np.asarray(gwr_residuals)
                gwr_beta_bse = gwr_results.bse
                gwr_r2 = gwr_results.localR2

                allklen_stwr = cal_X_list[-1].shape[1]+1
                stwr_cal_parmas = np.reshape(stwr_results.params.flatten(),(-1,allklen_stwr))
                gwr_cal_parmas  = np.reshape(gwr_results.params.flatten(),(-1,allklen_stwr))

                ##############################################################################
    #            output_filepath = "D:/STWR/Data_STWR/Bisai-Chivs/Results/"
                output_file_stamp = extract_str_filedate(file_loads[size_len])
                output_filename= output_filepath + dir +'/'+ output_file_stamp

                if os.path.exists(output_filename) == False:
                       os.makedirs(output_filename)

                data=open(output_filename+"/summary.txt",'w+')
                print(stwr_results.summary(),file=data)
                print(gwr_results.summary(),file=data)
                data.close()

                AICc_out_tick = np.array([get_AICc(ols_result),gwr_results.aicc,stwr_results.aicc,output_file_stamp])
                #AICc_out_tick = np.array([get_AICc(ols_result),stwr_results.aicc,output_file_stamp])
                AICc_out.append(AICc_out_tick)
                R2_out_tick = np.array([ols_result.D2,gwr_results.R2,stwr_results.R2,output_file_stamp])
                R2_out.append(R2_out_tick)
                RSS_out_tick = np.array([ols_result.deviance,gwr_results.resid_ss,stwr_results.resid_ss,output_file_stamp])
                RSS_out.append(RSS_out_tick)
                ols_sigma = np.sqrt(ols_result.deviance/cal_X_list[-1].shape[0]-cal_X_list[-1].shape[1]-1)
                Sigma_out_tick = np.array([ols_sigma,np.sqrt(gwr_results.sigma2),np.sqrt(stwr_results.sigma2),output_file_stamp])
                Sigma_out.append(Sigma_out_tick)

                ##############################################################################
    #            "%-62s %12.3f\n" %  ('Residual sum of squares:', glm_rslt.deviance)
    #            "%-62s %12.3f\n" %  ('Log-likelihood:', glm_rslt.llf)
    #            summary += "%-62s %12.3f\n" %  ('AIC:', glm_rslt.aic)
    #            summary += "%-62s %12.3f\n" %  ('AICc:', get_AICc(glm_rslt))
    #            summary += "%-62s %12.3f\n" %  ('BIC:', glm_rslt.bic)
    #            summary += "%-62s %12.3f\n" %  ('R2:', glm_rslt.D2)
    #            summary += "%-62s %12.3f\n\n" % ('Adj. R2:', glm_rslt.adj_D2)
                 #"%-31s %10.3f %10.3f %10.3f %10.3f\n" % (XNames[i], glm_rslt.params[i], glm_rslt.bse[i], glm_rslt.tvalues[i], glm_rslt.pvalues[i])
                 #ols.sigma = sqrt(glm_rslt.deviance/n-k-1)
                 #gwr
                 #summary += "%-62s %12.3f\n" % ('Bandwidth used:', self.model.bw)
    #            summary += "%-62s %12.3f\n" % ('Residual sum of squares:', self.resid_ss)
    #            summary += "%-62s %12.3f\n" % ('Effective number of parameters (trace(S)):', self.tr_S)
    #            summary += "%-62s %12.3f\n" % ('Degree of freedom (n - trace(S)):', self.df_model)
    #            summary += "%-62s %12.3f\n" % ('Sigma estimate:', np.sqrt(self.sigma2))
    #            summary += "%-62s %12.3f\n" % ('Log-likelihood:', self.llf)
    #            summary += "%-62s %12.3f\n" % ('AIC:', self.aic)
    #            summary += "%-62s %12.3f\n" % ('AICc:', self.aicc)
    #            summary += "%-62s %12.3f\n" % ('BIC:', self.bic)
    #            summary += "%-62s %12.3f\n" % ('R2:', self.R2)
                 #STWR
    #            summary += "%-62s %12.3f\n" % ('Residual sum of squares:', self.resid_ss)
    #            summary += "%-62s %12.3f\n" % ('Effective number of parameters (trace(S)):', self.tr_S)
    #            summary += "%-62s %12.3f\n" % ('Degree of freedom (n - trace(S)):', self.df_model)
    #            summary += "%-62s %12.3f\n" % ('Sigma estimate:', np.sqrt(self.sigma2))
    #            summary += "%-62s %12.3f\n" % ('Log-likelihood:', self.llf)
    #            summary += "%-62s %12.3f\n" % ('AIC:', self.aic)
    #            summary += "%-62s %12.3f\n" % ('AICc:', self.aicc)
    #            summary += "%-62s %12.3f\n" % ('BIC:', self.bic)
    #            summary += "%-62s %12.3f\n" % ('R2:', self.R2)

                #gwr_results.resid_ss

                df_stwr_resids = pd.DataFrame({
                                    'Ename':Ename,
                                    'lon':cal_cord_gwr[:,0],
                                    'lat':cal_cord_gwr[:,1],
                                    'resid':stwr_residuals
                                    })
                df_stwr_resids.to_csv(output_filename+"/stwr_residuals.csv")


                df_gwr_resids = pd.DataFrame({
                                         'Ename':Ename,
                                         'lon':cal_cord_gwr[:,0],
                                         'lat':cal_cord_gwr[:,1],
                                         'resid':gwr_residuals
                                       })
                df_gwr_resids.to_csv(output_filename+"/gwr_residuals.csv")

                df_stwr_r2 = pd.DataFrame({
                                    'Ename':Ename,
                                    'lon':cal_cord_gwr[:,0],
                                    'lat':cal_cord_gwr[:,1],
                                    'R2':stwr_r2[:,0]
                                    })
                df_stwr_r2.to_csv(output_filename+"/stwr_R2.csv")
                df_gwr_r2 = pd.DataFrame({
                                       'Ename':Ename,
                                       'lon':cal_cord_gwr[:,0],
                                       'lat':cal_cord_gwr[:,1],
                                       'R2':gwr_r2[:,0]
                                       })
                df_gwr_r2.to_csv(output_filename+"/gwr_R2.csv")

                # 	U_m_s_	V_m_s_	TEMP_K_	RH___	PSFC_Pa_
                df_stwr_parmas = pd.DataFrame({
                                        'Ename':Ename,
                                        'lon':cal_cord_gwr[:,0],
                                        'lat':cal_cord_gwr[:,1],
                                        'constant':stwr_cal_parmas[:,0],
                                        'Y1':stwr_cal_parmas[:,1],
                                        'Y2':stwr_cal_parmas[:,2],
                                        'Y3': stwr_cal_parmas[:, 3],
                                        'Y4': stwr_cal_parmas[:, 4],
                                        #'jyl': stwr_cal_parmas[:, 5],
                                        #'psfc': stwr_cal_parmas[:, 6],
                                        
                                         })
                df_stwr_parmas.to_csv(output_filename+"/stwr_parmas.csv")
                df_gwr_parmas = pd.DataFrame({
                                        'Ename':Ename,
                                        'lon':cal_cord_gwr[:,0],
                                        'lat':cal_cord_gwr[:,1],
                                        'constant':gwr_cal_parmas[:,0],
                                        #'o3':stwr_cal_parmas[:,1],
                                        'Y1':gwr_cal_parmas[:,1],
                                        #'psfc':stwr_cal_parmas[:,2],
                                        'Y2':gwr_cal_parmas[:,2],
                                        #'czhl':stwr_cal_parmas[:,5],
                                        'Y3': gwr_cal_parmas[:, 3],
                                        'Y4': gwr_cal_parmas[:, 4],
                                        #'rkmd': gwr_cal_parmas[:, 5],
                                        #'jyl': gwr_cal_parmas[:, 5],
                                        #'psfc': gwr_cal_parmas[:, 6],
                })

                df_gwr_parmas.to_csv(output_filename+"/gwr_parmas.csv")

                stwr_cal_tvalues = stwr_cal_parmas/stwr_beta_se
                gwr_cal_tvalues =  gwr_cal_parmas/gwr_beta_bse

                stwr_cal_pvalues = stats.t.sf(np.abs(stwr_cal_tvalues),stwr_results.df_model )*2
                gwr_cal_pvalues = stats.t.sf(np.abs(gwr_cal_tvalues),gwr_results.df_model )*2
                ##############################################################################

                df_stwr_tvalue = pd.DataFrame({
                                        'Ename':Ename,
                                        'lon':cal_cord_gwr[:,0],
                                        'lat':cal_cord_gwr[:,1],
                                        'constant':stwr_cal_parmas[:,0],
                                        #'o3':stwr_cal_parmas[:,1],
                                        'Y1':stwr_cal_parmas[:,1],
                                        #'psfc':stwr_cal_parmas[:,2],
                                        'Y2':stwr_cal_parmas[:,2],
                                        #'czhl':stwr_cal_parmas[:,5],
                                        'Y3': stwr_cal_parmas[:, 3],
                                        'Y4': stwr_cal_parmas[:, 4],
                                        #'rkmd': stwr_cal_parmas[:, 5],
                                        #'psfc': stwr_cal_parmas[:, 6],
                                        })
                df_stwr_tvalue.to_csv(output_filename+"/stwr_t-Values.csv")
                df_gwr_tvalue = pd.DataFrame({
                                        'Ename':Ename,
                                        'lon':cal_cord_gwr[:,0],
                                        'lat':cal_cord_gwr[:,1],
                                        'constant':gwr_cal_parmas[:,0],
                                        #'o3':stwr_cal_parmas[:,1],
                                        'Y1':gwr_cal_parmas[:,1],
                                        #'psfc':stwr_cal_parmas[:,2],
                                        'Y2':gwr_cal_parmas[:,2],
                                        #'czhl':stwr_cal_parmas[:,5],
                                        'Y3': gwr_cal_parmas[:, 3],
                                        'Y4': gwr_cal_parmas[:, 4],
                                        #'rkmd': gwr_cal_parmas[:, 5],
                                        #'psfc': gwr_cal_parmas[:, 6],
                })
                df_gwr_tvalue.to_csv(output_filename+"/gwr_t-Values.csv")

                df_stwr_pvalue = pd.DataFrame({
                                        'Ename':Ename,
                                        'lon':cal_cord_gwr[:,0],
                                        'lat':cal_cord_gwr[:,1],
                                        'constant':stwr_cal_parmas[:,0],
                                        #'o3':stwr_cal_parmas[:,1],
                                        'Y1':stwr_cal_parmas[:,1],
                                        #'psfc':stwr_cal_parmas[:,2],
                                        'Y2':stwr_cal_parmas[:,2],
                                        #'czhl':stwr_cal_parmas[:,5],
                                        'Y3': stwr_cal_parmas[:, 3],
                                        'Y4': stwr_cal_parmas[:, 4],
                                        #'rkmd': stwr_cal_parmas[:, 5],
                                        #'psfc': stwr_cal_parmas[:, 6],
                })
                df_stwr_pvalue.to_csv(output_filename+"/stwr_p-Values.csv")

                df_gwr_pvalue = pd.DataFrame({
                                        'Ename':Ename,
                                        'lon':cal_cord_gwr[:,0],
                                        'lat':cal_cord_gwr[:,1],
                                        'constant':gwr_cal_parmas[:,0],
                                        #'o3':stwr_cal_parmas[:,1],
                                        'Y1':gwr_cal_parmas[:,1],
                                        #'psfc':stwr_cal_parmas[:,2],
                                        'Y2':gwr_cal_parmas[:,2],
                                        #'czhl':stwr_cal_parmas[:,5],
                                        'Y3': gwr_cal_parmas[:, 3],
                                        'Y4': gwr_cal_parmas[:, 4],
                                        #'rkmd': gwr_cal_parmas[:, 5],
                                        #'psfc': gwr_cal_parmas[:, 6],
                })
                df_gwr_pvalue.to_csv(output_filename+"/gwr_p-Values.csv")
                ##############################################################################
                stwr_cal_mean = np.mean(stwr_cal_parmas, axis=0)
                stwr_cal_mean

                gwr_cal_mean = np.mean(gwr_cal_parmas, axis=0)
                gwr_cal_mean

                strw_cal_std = np.std(stwr_cal_parmas, axis=0)
                strw_cal_std

                grw_cal_std = np.std(gwr_cal_parmas, axis=0)
                grw_cal_std

                stwr_cal_parmas_zscore = np.zeros_like(stwr_cal_parmas)
                tick_num = 0
                for stwr_tick in stwr_cal_parmas:
                    stwr_cal_parmas_zscore[tick_num] = (stwr_tick - stwr_cal_mean)/strw_cal_std
                    tick_num += 1
                stwr_cal_parmas_zscore

                gwr_cal_parmas_zscore = np.zeros_like(gwr_cal_parmas)
                tick_num = 0
                for gwr_tick in gwr_cal_parmas:
                    gwr_cal_parmas_zscore[tick_num] = (gwr_tick - gwr_cal_mean)/grw_cal_std
                    tick_num += 1
                gwr_cal_parmas_zscore
                df_stwr_pz = pd.DataFrame({
                                        'Ename':Ename,
                                        'lon':cal_cord_gwr[:,0],
                                        'lat':cal_cord_gwr[:,1],
                                        'constant':stwr_cal_parmas[:,0],
                                        #'o3':stwr_cal_parmas[:,1],
                                        'Y1':stwr_cal_parmas[:,1],
                                        #'psfc':stwr_cal_parmas[:,2],
                                        'Y2':stwr_cal_parmas[:,2],
                                        #'czhl':stwr_cal_parmas[:,5],
                                        'Y3': stwr_cal_parmas[:, 3],
                                        'Y4': stwr_cal_parmas[:, 4],
                                        #'rkmd': stwr_cal_parmas[:, 5],
                                        #'psfc': stwr_cal_parmas[:, 6],
                                        })
                df_stwr_pz.to_csv( output_filename +"/stwr_z-Scores.csv")
                    stwr_cal_parmas_zscore[tick_num] = (stwr_tick - stwr_cal_mean)/strw_cal_std
                    tick_num += 1
                stwr_cal_parmas_zscore

                gwr_cal_parmas_zscore = np.zeros_like(gwr_cal_parmas)
                tick_num = 0
                for gwr_tick in gwr_cal_parmas:
                    gwr_cal_parmas_zscore[tick_num] = (gwr_tick - gwr_cal_mean)/grw_cal_std
                    tick_num += 1
                gwr_cal_parmas_zscore
                df_stwr_pz = pd.DataFrame({
                                        'Ename':Ename,
                                        'lon':cal_cord_gwr[:,0],
                                        'lat':cal_cord_gwr[:,1],
                                        'constant':stwr_cal_parmas[:,0],
                                        #'o3':stwr_cal_parmas[:,1],
                                        'Y1':stwr_cal_parmas[:,1],
                df_gwr_pz = pd.DataFrame({
                                        'Ename':Ename,
                                        'lon':cal_cord_gwr[:,0],
                                        'lat':cal_cord_gwr[:,1],
                                        'constant':gwr_cal_parmas[:,0],
                                        #'o3':stwr_cal_parmas[:,1],
                                        'Y1':gwr_cal_parmas[:,1],
                                        #'psfc':stwr_cal_parmas[:,2],
                                        'Y2':gwr_cal_parmas[:,2],
                                        #'czhl':stwr_cal_parmas[:,5],
                                        'Y3': gwr_cal_parmas[:, 3],
                                        'Y4': gwr_cal_parmas[:, 4],
                                        #'rkmd': gwr_cal_parmas[:, 5],
                                        #'psfc': gwr_cal_parmas[:, 6],
                })
                df_gwr_pz.to_csv(output_filename+"/gwr_z-Scores.csv")
                size_len +=1




    AICc_out = np.asarray(AICc_out)
    df_AICc_out = pd.DataFrame({
                            'ols':AICc_out[:,0],
                            'gwr':AICc_out[:,1],
                            'stwr':AICc_out[:,2],
                            'date':AICc_out[:,3],
                            })
    df_AICc_out.to_csv(output_filepath +"AICc.csv")

    R2_out = np.asarray(R2_out)
    df_R2_out = pd.DataFrame({
                            'ols':R2_out[:,0],
                            'gwr':R2_out[:,1],
                            'stwr':R2_out[:,2],
                            'date':R2_out[:,3],
                            })
    df_R2_out.to_csv(output_filepath +"R2.csv")

    RSS_out = np.asarray(RSS_out)
    df_RSS_out = pd.DataFrame({
                            'ols':RSS_out[:,0],
                            'gwr':RSS_out[:,1],
                            'stwr':RSS_out[:,2],
                            'date':RSS_out[:,3],
                            })
    df_RSS_out.to_csv(output_filepath +"RSS.csv")

    Sigma_out =np.asarray(Sigma_out)
    df_Sigma_out= pd.DataFrame({
                            'ols':Sigma_out[:,0],
                            'gwr':Sigma_out[:,1],
                            'stwr':Sigma_out[:,2],
                            'date':Sigma_out[:,3],
                            })
    df_Sigma_out.to_csv(output_filepath +"Sigma.csv")
    
    
    
